# Recommendation Layer for Port Congestion Risk

This notebook implements the final stage of the congestion risk framework.

Earlier stages of the system performed the following tasks:

- Data cleaning and preparation  
- Feature engineering  
- Logistic regression model for congestion prediction  
- Cause analysis to identify operational drivers of congestion  

The goal of this stage is to convert these signals into practical logistics decisions.

The recommendation engine evaluates:

- congestion risk  
- operational causes of congestion  
- remaining time before the demurrage threshold  

Based on these factors, the system recommends the most appropriate operational response for a logistics planner.
 

## Importing Libraries

In [1]:
import numpy as np
import pandas as pd

## Load Cause Analysis Dataset

This dataset is generated by the previous stage of the framework.

It contains the following important fields:

- Risk – congestion indicator from the ML model  
- Primary_Explanation – main operational driver of congestion  
- Secondary_Explanation – supporting congestion signal  

These signals will be used by the recommendation engine.

In [2]:
df_rec =pd.read_csv("../../data/JNPA_cause_analysis_output.csv")
df_rec.head()

,Month-Year,Imp_Dwell_Overall,Imp_Dwell_Truck,Imp_Dwell_Train,Parking_Dwell,Imp_Transit_CFS,Imp_Transit_ICD,Exp_Dwell_Overall,Exp_Dwell_Truck,Exp_Dwell_Train,...,Date,Volume_Lag1,Rail_Friction,Is_Monsoon,Stress,Risk,Primary_Cause,Secondary_Cause,Primary_Explanation,Secondary_Explanation
0,Feb 23,27.8,24.0,69.6,2.55,2.78,79.0,69.5,67.8,80.7,...,2023-02-01,522592.0,2.900000,0,0,1,Parking_Dwell,NaN,Export gate congestion,NaN
1,Mar 23,26.5,22.3,63.7,3.65,2.47,97.1,74.0,72.4,86.1,...,2023-03-01,467614.0,2.856502,0,0,0,Parking_Dwell,NaN,Export gate congestion,NaN
2,Apr 23,29.9,25.4,53.2,4.00,2.62,90.1,80.0,77.1,104.6,...,2023-04-01,560076.0,2.094488,0,0,1,Parking_Dwell,NaN,Export gate congestion,NaN
3,May 23,19.9,17.2,51.0,2.33,2.56,91.1,65.0,62.8,81.3,...,2023-05-01,503668.0,2.965116,0,0,0,Parking_Dwell,NaN,Export gate congestion,NaN
4,Jun 23,15.8,13.8,38.4,2.18,2.35,107.0,71.9,69.8,88.5,...,2023-06-01,538247.0,2.782609,1,523948,0,Stress,NaN,Operational throughput pressure,NaN


## Port Operational Parameters

Most Indian ports including JNPA allow containers to remain in the yard for approximately 72 hours of free time before demurrage charges begin.

Demurrage charges increase progressively once the free period expires.

For this simplified financial model we assume:

 Free Time = 72 hours

In [3]:
Free_Time = 72

## Planner Inputs

Instead of entering dwell time manually, the planner selects the month
for which the logistics decision is being evaluated.

The system retrieves operational indicators for that month from the dataset.

Inputs required:

- shipment type  
- transport mode  
- destination type  
- number of containers (TEUs)  
- month of operation

In [4]:
print("Enter logistics planning scenario")

shipment_type = input("Shipment Type (Import/Export): ")

transport_mode = input("Transport Mode (Truck/Rail): ")

destination_type = input("Destination Type (NearPort/Inland): ")

teu_count = int(input("Number of containers (TEUs): "))

month_selected = input("Enter month (e.g., Feb 23): ")   

Enter logistics planning scenario


## Retrieve Operational Data

The system retrieves congestion indicators and dwell time
for the selected month from the dataset.

In [5]:
selected_row = df_rec[df_rec["Month-Year"] == month_selected]

if selected_row.empty:
    print("Month not found in dataset")
else:
    selected_row = selected_row.iloc[0]

risk = selected_row["Risk"]

primary_cause = selected_row["Primary_Explanation"]

secondary_cause = selected_row["Secondary_Explanation"]

current_dwell = selected_row["Imp_Dwell_Overall"]

## Remaining Free Time

Remaining free time is calculated using the dwell time recorded
for the selected month.

In [6]:
remaining_time = Free_Time - current_dwell

print("Current Dwell Time:", current_dwell)

print("Remaining Free Time:", remaining_time)

Current Dwell Time: 25.3
Remaining Free Time: 46.7


## Buffer Classification

Remaining time is grouped into three zones to simplify decisions.

Safe Zone - more than 24 hours remaining  
Warning Zone - between 12 and 24 hours remaining  
Critical Zone - less than 12 hours remaining

In [7]:
if remaining_time > 24:
    buffer_zone = "Safe"
elif remaining_time > 12:
    buffer_zone = "Warning"
else:
    buffer_zone = "Critical"

print("Buffer Zone:", buffer_zone)

Buffer Zone: Safe


## Congestion Signals

The latest row of the dataset represents the most recent operational state of the port system.

In [8]:
latest = df_rec.iloc[-1]

risk = latest["Risk"]
primary_cause = latest["Primary_Explanation"]
secondary_cause = latest["Secondary_Explanation"]

print("Congestion Risk:", risk)
print("Primary Cause:", primary_cause)
print("Secondary Cause:", secondary_cause)

Congestion Risk: 1
Primary Cause: High container demand
Secondary Cause: nan


## Estimated Congestion Delay

If congestion risk is detected, a small operational delay is assumed.

This simplified model assumes a delay of two days when congestion occurs.

In [9]:
if risk == 1:
    estimated_delay_days = 2
else:
    estimated_delay_days = 0

## Recommendation Engine

The system combines:

- congestion risk  
- operational cause of congestion  
- remaining free time  
- estimated financial costs  

to determine the most appropriate action.

In [10]:
recommendation = "Monitor port conditions."

if risk == 1:

    if buffer_zone == "Critical":

        if primary_cause == "Export gate congestion":
            recommendation = "Delay truck entry or adjust gate arrival schedules."

        elif primary_cause == "Rail evacuation imbalance":
            recommendation = "Switch evacuation to truck if possible."

        elif primary_cause == "Operational throughput pressure":
            recommendation = "Schedule container evacuation earlier."

        elif primary_cause == "High container demand":
            recommendation = "Spread container pickups across multiple days."

        else:
            recommendation = "Review port congestion conditions."

    elif buffer_zone == "Warning":
        recommendation = "Prepare evacuation plan soon."

    else:
        recommendation = "Congestion detected but sufficient time buffer remains."

## Destination Adjustment

For inland shipments, rail evacuation through an ICD may be considered when congestion risk is high.

In [11]:
if destination_type == "Inland" and primary_cause == "Rail evacuation imbalance":
    recommendation += " Consider alternate rail slot or ICD routing."

## Logistics Decision Summary

In [12]:
print("\n--- Logistics Decision Summary ---")

print("Selected Month:", month_selected)

print("Shipment Type:", shipment_type)
print("Transport Mode:", transport_mode)
print("Destination:", destination_type)

print("\nCongestion Risk:", risk)

print("\nCause Analysis")
print("Primary Cause:", primary_cause)
print("Secondary Cause:", secondary_cause)

print("\nRecommendation")
print(recommendation)


--- Logistics Decision Summary ---
Selected Month: Oct 25
Shipment Type: Import
Transport Mode: Rail
Destination: Inland

Congestion Risk: 1

Cause Analysis
Primary Cause: High container demand
Secondary Cause: nan

Recommendation
Congestion detected but sufficient time buffer remains.


## Recommendation saving 

In [13]:
df_output = pd.DataFrame({
    "Month": [month_selected],
    "Shipment_Type": [shipment_type],
    "Transport_Mode": [transport_mode],
    "Destination": [destination_type],
    "Risk": [risk],
    "Primary_Cause": [primary_cause],
    "Secondary_Cause": [secondary_cause],
    "Recommendation": [recommendation]
})

df_output.to_csv("../../data/JNPA_recommendations.csv", index=False)

## Financial Impact Estimation (Future Work)

A financial estimation module can be integrated into the framework to estimate potential demurrage exposure.

Such a module would combine:
- predicted congestion risk
- estimated dwell time
- container count (TEUs)
- demurrage tariff structures defined by port authorities

Due to the simplified nature of the current dataset and the absence of precise dwell time prediction, the financial estimation component is not implemented in this prototype.

The current framework focuses on identifying congestion risk and operational drivers, enabling logistics planners to take preventive action before demurrage thresholds are reached.